In [1]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
  %cd /content/drive/MyDrive/SnowPole_Detection_Dataset/
# !git clone https://github.com/ultralytics/ultralytics.git

/content/drive/.shortcut-targets-by-id/1ofcfwBISHeUE4V1x6Bj-J5Mw3-QSUHFB/SnowPole_Detection_Dataset


In [3]:
!pip uninstall -y ultralytics

Found existing installation: ultralytics 8.2.5
Uninstalling ultralytics-8.2.5:
  Successfully uninstalled ultralytics-8.2.5


In [4]:
!rm -rf /content/ultralytics
!git clone https://github.com/g-h-anna/ultralytics4channel.git /content/ultralytics

Cloning into '/content/ultralytics'...
remote: Enumerating objects: 263, done.
remote: Counting objects: 100% (263/263), done.
remote: Compressing objects: 100% (219/219), done.
remote: Total 263 (delta 53), reused 243 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (263/263), 701.94 KiB | 4.44 MiB/s, done.
Resolving deltas: 100% (53/53), done.


In [5]:
!pip install -q ultralytics==8.2.5

In [6]:
import sys
sys.path.insert(0, "/content")  # parent of the ultralytics package

import ultralytics
from ultralytics import YOLO

print("Ultralytics module file:", ultralytics.__file__)


Ultralytics module file: /content/ultralytics/__init__.py


In [12]:
%cd /content/ultralytics4channel

!python -m ultralytics train \
  model="/content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/yolov9t_dual.yaml" \
  data="/content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/data.yaml" \
  epochs=400 imgsz=1024 device=cpu batch=16 \
  project="dual_comb_range_experiments" \
  name="dual_comb_rgb_plus_range_v9t_4ch_825"


/content/ultralytics4channel
/usr/bin/python3: No module named ultralytics.__main__; 'ultralytics' is a package and cannot be directly executed


In [7]:
import torch
from ultralytics import YOLO
from pathlib import Path
import shutil
import cv2
import numpy as np
import yaml
from tqdm import tqdm

from ultralytics import YOLO
import torch

from torch.utils.data import Dataset, DataLoader

In [8]:
COMB_ROOT  = Path("/content/drive/MyDrive/SnowPole_Detection_Dataset/images")

# 1-channel range-normalized images
RANGE_ROOT = Path("/content/drive/MyDrive/SnowPole_Detection_Dataset/range-normalized-continuous")

# New 4-channel dual-input dataset
DUAL_ROOT  = Path("/content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range")
DUAL_ROOT.mkdir(parents=True, exist_ok=True)

print("COMB_ROOT :", COMB_ROOT)
print("RANGE_ROOT:", RANGE_ROOT)
print("DUAL_ROOT :", DUAL_ROOT)

COMB_ROOT : /content/drive/MyDrive/SnowPole_Detection_Dataset/images
RANGE_ROOT: /content/drive/MyDrive/SnowPole_Detection_Dataset/range-normalized-continuous
DUAL_ROOT : /content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range


In [ ]:
# !yolo train model=yolov9t.pt epochs=150 imgsz=1024 device=0 batch=2 data=/content/drive/MyDrive/data.yaml project=/content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec

In [ ]:
# !yolo val \
#   model=/content/drive/.shortcut-targets-by-id/1ofcfwBISHeUE4V1x6Bj-J5Mw3-QSUHFB/SnowPole_Detection_Dataset/comb4-range-signal-reflec-v11n/train2/weights/best.pt \
#   data=/content/drive/MyDrive/data.yaml \
#   split=test \
#   imgsz=1024 \
#   device=0 \
#   batch=16 \
#   project="comb5_signal_reflec_range_11n" \
#   name="comb5_signal_reflec_range_11n_test_eval"


In [9]:
def make_dual_split_npy(split: str,
                        comb_root: Path,
                        range_root: Path,
                        dual_root: Path,
                        save_as_png: bool = True):
    """
    Create 4-channel dual images by stacking comb RGB (BGR) + range (.npy float32 [0,1]).
    - comb_root: root containing comb_root/images/<split>/*.png and comb_root/labels/<split>/*.txt
    - range_root: root containing range_root/<split>/*.npy (each named like the comb image stem)
    - dual_root: destination root; will create dual_root/images/<split> and dual_root/labels/<split>
    - save_as_png: if True, save stacked RGBA PNGs (4 channel) so existing YOLO loaders can read them.
                   (range channel is quantized to uint8 for the PNG; the original .npy is left unchanged)
    """
    comb_img_dir   = comb_root / split
    src_lbl_dir    = comb_root / "../" / "labels" / split
    range_npy_dir  = range_root / split
    dual_img_dir   = dual_root / "images" / split
    dual_lbl_dir   = dual_root / "labels" / split

    # if this split already has images, skip doing anything
    if dual_img_dir.exists() and any(dual_img_dir.glob("*.png")):
        print(f"[{split}] dual images already exist in {dual_img_dir}, skipping.")
        return

    dual_img_dir.mkdir(parents=True, exist_ok=True)
    dual_lbl_dir.mkdir(parents=True, exist_ok=True)

    # copy labels from comb labels to dual labels (they are the same)
    if src_lbl_dir.exists():
        for lbl in src_lbl_dir.glob("*.txt"):
            # copy2 preserves timestamps, etc.
            shutil.copy2(lbl, dual_lbl_dir / lbl.name)
    else:
        print(f"Warning: source label dir not found: {src_lbl_dir}")

    # gather comb images
    img_files = sorted(comb_img_dir.glob("*.*"))
    print(f"[{split}] comb images found: {len(img_files)}")

    for comb_path in tqdm(img_files, desc=f"make_dual_split ({split})"):
        stem = comb_path.stem

        # read comb RGB (OpenCV: BGR)
        comb = cv2.imread(str(comb_path), cv2.IMREAD_COLOR)
        if comb is None:
            print("Could not read comb image:", comb_path)
            continue

        # read corresponding range .npy
        range_npy_path = range_npy_dir / f"{stem}.npy"
        if not range_npy_path.exists():
            # try alternative stem patterns if needed
            print(f"Missing range .npy for {stem} -> {range_npy_path} (skipping)")
            continue

        try:
            range_arr = np.load(str(range_npy_path))   # expected float32 in [0,1]
        except Exception as e:
            print(f"Failed to load {range_npy_path}: {e}")
            continue

        # squeeze any extra dims
        if range_arr.ndim == 3 and range_arr.shape[0] in (1,):
            range_arr = np.squeeze(range_arr, axis=0)
        if range_arr.ndim != 2:
            # if it has a channel dim (H,W,1) -> squeeze
            if range_arr.ndim == 3 and range_arr.shape[2] == 1:
                range_arr = np.squeeze(range_arr, axis=2)
            else:
                print(f"Unexpected shape for range npy {range_npy_path}: {range_arr.shape} (skipping)")
                continue

        # ensure float32 and clip to [0,1]
        range_arr = range_arr.astype(np.float32)
        range_arr = np.clip(range_arr, 0.0, 1.0)

        # convert range to uint8 for stacking if saving PNGs (visualization/training with standard loader)
        range_uint8 = (range_arr * 255.0).astype(np.uint8)

        # resize range to comb dims if necessary (note cv2 resize expects (width, height))
        if range_uint8.shape != comb.shape[:2]:
            range_uint8 = cv2.resize(range_uint8, (comb.shape[1], comb.shape[0]), interpolation=cv2.INTER_NEAREST)

        # stack into 4-channel: B, G, R, RANGE
        rgba = np.dstack([comb, range_uint8])  # result dtype uint8, shape (H, W, 4)

        # write stacked 4-channel PNG so YOLO-like image loaders can ingest it
        if save_as_png:
            out_img_path = dual_img_dir / f"{stem}.png"
            # OpenCV will write all 4 channels to PNG when given a 4-channel array.
            cv2.imwrite(str(out_img_path), rgba)

    print(f"[{split}] done. Dual images written to {dual_img_dir}, labels copied to {dual_lbl_dir}")


# Run for all splits (call this cell)
for split in ["train", "valid", "test"]:
    make_dual_split_npy(split, comb_root=COMB_ROOT, range_root=RANGE_ROOT, dual_root=DUAL_ROOT, save_as_png=True)


[train] dual images already exist in /content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/images/train, skipping.
[valid] dual images already exist in /content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/images/valid, skipping.
[test] dual images already exist in /content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/images/test, skipping.


In [15]:
model = YOLO("yolov9t.pt")           # Step 2: load pretrained RGB weights
model.model.eval()                   # get the underlying nn.Module graph

# first_conv = model.model.model[0]    # YOLOv9 stem (Conv → BN → SILU)
# old_conv = first_conv.conv
# new_conv = torch.nn.Conv2d(
#     in_channels=4,
#     out_channels=old_conv.out_channels,
#     kernel_size=old_conv.kernel_size,
#     stride=old_conv.stride,
#     padding=old_conv.padding,
#     bias=old_conv.bias is not None,
# )

# with torch.no_grad():
#     new_conv.weight[:, :3] = old_conv.weight            # copy RGB kernels
#     new_conv.weight[:, 3:] = old_conv.weight[:, :1] * 0 # or torch.randn_like(...)*1e-3
#     if old_conv.bias is not None:
#         new_conv.bias = old_conv.bias

# first_conv.conv = new_conv            # swap into the module tree
# model.model.model[0] = first_conv     # ensure the model graph is updated

# model.save("yolov9t_rgba.pt")
model.train(
    data=str(DUAL_DATA_YAML),
    epochs=400,
    imgsz=1024,
    device="cpu",
    batch=16,
    project="dual_comb_range_experiments",
    name="dual_comb_rgb_plus_range_v9t_4ch_825",
)


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL ultralytics.nn.tasks.DetectionModel was not an allowed global by default. Please use `torch.serialization.add_safe_globals([ultralytics.nn.tasks.DetectionModel])` or the `torch.serialization.safe_globals([ultralytics.nn.tasks.DetectionModel])` context manager to allowlist this global if you trust this class/function.

Check the documentation of to

In [ ]:
# ORIG_DATA_YAML = COMB_ROOT / "../" /"data.yaml"
# DUAL_DATA_YAML = DUAL_ROOT / "data.yaml"

# with open(ORIG_DATA_YAML, "r") as f:
#     cfg = yaml.safe_load(f)

# base = DUAL_ROOT

# def make_rel(p):
#     # p might be absolute or relative – we point to new dual root
#     p = Path(p)
#     return str((base / "images" / p.name).parent)  # keep split names

# # If your original yaml used explicit paths, you can instead do:
# # cfg["path"]  = str(DUAL_ROOT)
# cfg["path"]  = str(DUAL_ROOT)
# cfg["train"] = "images/train"
# cfg["valid"]   = "images/valid"
# cfg["test"]  = "images/test"
# cfg["channels"] = 4          # tell YOLO this is 4-channel data with the RGB-Alpha

# with open(DUAL_DATA_YAML, "w") as f:
#     yaml.safe_dump(cfg, f)

# print(DUAL_DATA_YAML.read_text())

In [10]:
# 1. Define where the dual model yaml will live
DUAL_MODEL_YAML = DUAL_ROOT / "yolov9t_dual.yaml"
print("DUAL_MODEL_YAML:", DUAL_MODEL_YAML)

# # 2. Download official yolov9t.yaml from Ultralytics repo into that path
# !wget -O "{DUAL_MODEL_YAML}" https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/cfg/models/v9/yolov9t.yaml

# # 3. Load, patch ch and nc, and save back
# cfg["ch"] = 4          # 4 input channels (RGB + range)
# cfg = yaml.safe_load(DUAL_MODEL_YAML.read_text())
# cfg["nc"] = 1          # your number of classes

# # DUAL_MODEL_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False))
# print("Final yolov9t_dual.yaml content:\n")
# print(DUAL_MODEL_YAML.read_text())

DUAL_MODEL_YAML: /content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/yolov9t_dual.yaml


In [12]:
# from ultralytics import YOLO
# import torch

# # Load model from yaml
# four_ch_model = YOLO(str(DUAL_MODEL_YAML))
# net = four_ch_model.model

# # Replace first conv with 4‑channel version
# first_block = net.model[0]
# old_conv = first_block.conv
# print("OLD conv:", old_conv.weight.shape)  # expect [16, 3, 3, 3]

# new_conv = torch.nn.Conv2d(
#     in_channels=4,
#     out_channels=old_conv.out_channels,
#     kernel_size=old_conv.kernel_size,
#     stride=old_conv.stride,
#     padding=old_conv.padding,
#     bias=(old_conv.bias is not None),
# )

# with torch.no_grad():
#     new_conv.weight[:, :3, :, :] = old_conv.weight
#     new_conv.weight[:, 3:, :, :] = 0.0
#     if old_conv.bias is not None:
#         new_conv.bias.copy_(old_conv.bias)

# first_block.conv = new_conv
# net.model[0] = first_block
# four_ch_model.model = net

# print("NEW conv:", four_ch_model.model.model[0].conv.weight.shape)  # must be [16, 4, 3, 3]


KeyError: 'ELAN1'

In [ ]:
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import cv2
import numpy as np
import torch

class DualYOLODataset(Dataset):
    def __init__(self, images_dir: Path, labels_dir: Path, img_size=1024):
        self.images = sorted(images_dir.glob("*.png"))
        self.labels_dir = labels_dir
        self.img_size = img_size

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)  # keep 4 channels
        assert img is not None, f"Failed to read {img_path}"

        img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))  # 4 x H x W

        stem = img_path.stem
        label_path = self.labels_dir / f"{stem}.txt"
        if label_path.exists():
            targets = np.loadtxt(str(label_path), ndmin=2).astype(np.float32)  # (N,5)
        else:
            targets = np.zeros((0, 5), dtype=np.float32)

        sample = {
            "img": torch.from_numpy(img),
            "cls": torch.from_numpy(targets[:, 0:1]) if targets.size else torch.zeros((0, 1), dtype=torch.float32),
            "bboxes": torch.from_numpy(targets[:, 1:5]) if targets.size else torch.zeros((0, 4), dtype=torch.float32),
            "im_file": str(img_path),
        }
        return sample


In [ ]:
from torch.utils.data import DataLoader

train_images = DUAL_ROOT / "images" / "train"
train_labels = DUAL_ROOT / "labels" / "train"

train_dataset = DualYOLODataset(train_images, train_labels, img_size=1024)
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    collate_fn=lambda x: x,
)

device = torch.device("cpu")
four_ch_model.model.to(device)
four_ch_model.model.train()

optimizer = torch.optim.AdamW(four_ch_model.model.parameters(), lr=2e-3)
num_epochs = 400

for epoch in range(num_epochs):
    for batch in train_loader:
        imgs = torch.stack([b["img"] for b in batch], dim=0).to(device)  # [B, 4, H, W]

        cls_list = [b["cls"] for b in batch]
        box_list = [b["bboxes"] for b in batch]

        batch_idx_list = []
        targets_list = []
        for i, (cls_i, box_i) in enumerate(zip(cls_list, box_list)):
            if cls_i.numel() == 0:
                continue
            n = cls_i.shape[0]
            bi = torch.full((n, 1), i, dtype=torch.float32)
            targets_list.append(torch.cat([bi, cls_i, box_i], dim=1))  # (n,6)

        if len(targets_list):
            targets = torch.cat(targets_list, dim=0)  # (N,6)
            batch_idx_tensor = targets[:, 0].to(device)
            cls_tensor = targets[:, 1:2].to(device)
            bboxes_tensor = targets[:, 2:6].to(device)
        else:
            batch_idx_tensor = torch.zeros((0,), dtype=torch.float32, device=device)
            cls_tensor = torch.zeros((0, 1), dtype=torch.float32, device=device)
            bboxes_tensor = torch.zeros((0, 4), dtype=torch.float32, device=device)

        yolo_batch = {
            "img": imgs,
            "batch_idx": batch_idx_tensor,
            "cls": cls_tensor,
            "bboxes": bboxes_tensor,
            "im_file": [b["im_file"] for b in batch],
        }

        optimizer.zero_grad()
        loss, loss_items = four_ch_model.model(yolo_batch)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}/{num_epochs} - loss: {loss.item():.4f}")


In [ ]:
#prev one

# train_images = DUAL_ROOT / "images" / "train"
# train_labels = DUAL_ROOT / "labels" / "train"

# train_dataset = DualYOLODataset(train_images, train_labels, img_size=1024)
# train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,
#                           num_workers=0, collate_fn=lambda x: x)

# device = torch.device("cpu")
# four_ch_model.model.to(device)
# four_ch_model.model.train()

# optimizer = torch.optim.AdamW(four_ch_model.model.parameters(), lr=2e-3)
# num_epochs = 400

# #     for batch in train_loader:
# #         imgs = torch.stack([b["img"] for b in batch], dim=0).to(device)  # [B, 4, H, W]

# #         yolo_batch = {
# #             "img": imgs,
# #             "cls": [b["cls"].to(device) for b in batch],
# #             "bboxes": [b["bboxes"].to(device) for b in batch],
# #             "im_file": [b["im_file"] for b in batch],
# #         }

# #         optimizer.zero_grad()
# #         loss, loss_items = four_ch_model.model(yolo_batch)
# #         loss.backward()
# #         optimizer.step()

# #     print(f"Epoch {epoch+1}/{num_epochs} - loss: {loss.item():.4f}")
# for epoch in range(num_epochs):
#     for batch_i, batch in enumerate(train_loader):
#         imgs = torch.stack([b["img"] for b in batch], dim=0).to(device)  # [B, 4, H, W]
#         B = imgs.shape[0]

#         cls_list = [b["cls"] for b in batch]       # each (Ni,1)
#         box_list = [b["bboxes"] for b in batch]    # each (Ni,4)

#         # Build batch_idx vector and concat targets like Ultralytics expects
#         batch_idx = []
#         targets_list = []
#         for i, (cls_i, box_i) in enumerate(zip(cls_list, box_list)):
#             if cls_i.numel() == 0:
#                 continue
#             n = cls_i.shape[0]
#             bi = torch.full((n, 1), i, dtype=torch.float32)
#             targets_list.append(torch.cat([bi, cls_i, box_i], dim=1))  # (n,6)
#             batch_idx.append(bi)

#         if len(targets_list):
#             targets = torch.cat(targets_list, dim=0)       # (N,6): [batch, cls, x,y,w,h]
#             batch_idx_tensor = targets[:, 0].to(device)
#             cls_tensor = targets[:, 1:2].to(device)
#             bboxes_tensor = targets[:, 2:6].to(device)
#         else:
#             # no objects in batch
#             batch_idx_tensor = torch.zeros((0,), dtype=torch.float32, device=device)
#             cls_tensor = torch.zeros((0, 1), dtype=torch.float32, device=device)
#             bboxes_tensor = torch.zeros((0, 4), dtype=torch.float32, device=device)

#         yolo_batch = {
#             "img": imgs,
#             "batch_idx": batch_idx_tensor,
#             "cls": cls_tensor,
#             "bboxes": bboxes_tensor,
#             "im_file": [b["im_file"] for b in batch],
#         }

#         optimizer.zero_grad()
#         loss, loss_items = four_ch_model.model(yolo_batch)
#         loss.backward()
#         optimizer.step()

#     print(f"Epoch {epoch+1}/{num_epochs} - loss: {loss.item():.4f}")


In [ ]:
# # !yolo train \
# #   model="yolov9t_dual_4ch.pt" \
# #   data="{DUAL_DATA_YAML}" \
# #   epochs=400 imgsz=1024 device=cpu batch=16 name="dual_comb_rgb_plus_range_9t" project="dual_comb_range_experiments"


# # !yolo train \
# #   model="yolov9t_dual_4ch.pt" \
# #   data="{str(DUAL_DATA_YAML)}" \
# #   epochs=400 imgsz=1024 device=cpu batch=16 \
# #   name="dual_comb_rgb_plus_range_9t" \
# #   project="dual_comb_range_experiments"
# !yolo train \
#   model="{str(DUAL_MODEL_YAML)}" \
#   data="{str(DUAL_DATA_YAML)}" \
#   epochs=400 imgsz=1024 device=cpu batch=16 \
#   name="dual_comb_rgb_plus_range_9t4" \
#   project="dual_comb_range_experiments"


